# Torreta CIWS con IA: entrenamiento del clasificador de figuras

**Inteligencia Artificial en Dispositivos Móviles y Embebidos, Miniproyecto 3**
Juan Camilo Díaz Rangel (2215469)

Se entrena un clasificador **MobileNetV2** (transfer learning + fine-tuning) con un dataset **propio** capturado con la misma cámara de la torreta (`software/capturar_dataset.py`). Después se convierte a **TensorFlow Lite** cuantizado para correrlo en la Raspberry Pi 4.

Clases: `triangulo`, `cuadrado`, `circulo`, `estrella`, `cruz` (las 5 obligatorias) y, opcional, `otro` (lo que no es objetivo).

Se partió del notebook de clasificación de tapabocas visto en clase y se le hicieron estas correcciones:

| Problema en el notebook base | Corrección |
|---|---|
| `steps_per_epoch=len(train)//64`: solo 2 lotes por época (~128 de 10 000 imágenes) | Se recorre el dataset completo en cada época |
| Entrena en RGB y predice con `cv2.imread` en BGR sin convertir | Se convierte BGR→RGB, igual que en la Raspberry, y se valida el `.tflite` con ese mismo flujo |
| La normalización (`/255`) está por fuera del modelo, y es fácil olvidarla o hacerla distinta | El `Rescaling` va **dentro** del modelo: la Pi solo le pasa la imagen en 0-255 |
| Aumentación también en validación; `rotation_range=0.2` (0,2°) | Aumentación solo en entrenamiento, con rotación completa (las figuras pueden aparecer giradas) |
| Sin matriz de confusión, sin curvas, sin prueba del `.tflite` | Curvas, reporte por clase, matriz de confusión, exactitud y latencia del `.tflite` |

## 1. Configuración

In [ ]:
import os, glob, random, shutil, time, zipfile
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow", tf.__version__)
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

IMG = 160          # entrada del modelo (160x160): buen equilibrio entre exactitud y velocidad en la Pi 4
LOTE = 32
EPOCAS_1 = 15      # fase 1: solo la cabeza
EPOCAS_2 = 15      # fase 2: fine-tuning
FIGURAS = "/content/figuras"   # gráficas para el informe
os.makedirs(FIGURAS, exist_ok=True)

## 2. Dataset
Sube a tu Google Drive el archivo `dataset_torreta.zip`, la carpeta `dataset/` que generó `capturar_dataset.py` en la Raspberry, con una subcarpeta por clase.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/dataset_torreta.zip'   # cambia la ruta si lo guardaste en otra carpeta

shutil.rmtree('/content/crudo', ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/crudo')

# Busca la carpeta que contiene las subcarpetas de clases
RAIZ = '/content/crudo'
while True:
    subs = [d for d in os.listdir(RAIZ) if os.path.isdir(os.path.join(RAIZ, d)) and not d.startswith(('.', '__'))]
    if len(subs) == 1 and not any(f.lower().endswith(('.jpg', '.png', '.jpeg')) for f in os.listdir(os.path.join(RAIZ, subs[0]))):
        RAIZ = os.path.join(RAIZ, subs[0])
    else:
        break
CLASES = sorted(subs)
conteo = {c: len(glob.glob(os.path.join(RAIZ, c, '*'))) for c in CLASES}
print("Clases:", CLASES)
print("Imágenes por clase:", conteo, "| total:", sum(conteo.values()))
assert len([c for c in CLASES if c != 'otro']) >= 5, "La rúbrica pide al menos 5 clases"

In [ ]:
# Ejemplos de cada clase (figura para el informe)
fig, ax = plt.subplots(len(CLASES), 6, figsize=(12, 2.1 * len(CLASES)))
for i, c in enumerate(CLASES):
    archivos = random.sample(glob.glob(os.path.join(RAIZ, c, '*')), 6)
    for j, a in enumerate(archivos):
        ax[i, j].imshow(cv2.cvtColor(cv2.imread(a), cv2.COLOR_BGR2RGB)); ax[i, j].axis('off')
    ax[i, 0].set_title(c, loc='left', fontsize=11)
plt.tight_layout(); plt.savefig(f'{FIGURAS}/ejemplos_dataset.png', dpi=150); plt.show()

plt.figure(figsize=(7, 3))
plt.bar(conteo.keys(), conteo.values(), color='#3b6ea5')
plt.ylabel('imágenes'); plt.title('Imágenes por clase')
plt.tight_layout(); plt.savefig(f'{FIGURAS}/distribucion_clases.png', dpi=150); plt.show()

## 3. División entrenamiento / validación / prueba (70 / 15 / 15, por clase)

In [ ]:
DIV = '/content/division'
shutil.rmtree(DIV, ignore_errors=True)
for c in CLASES:
    archivos = sorted(glob.glob(os.path.join(RAIZ, c, '*')))
    random.shuffle(archivos)
    n = len(archivos); n_tr, n_va = int(0.70 * n), int(0.15 * n)
    partes = {'train': archivos[:n_tr], 'val': archivos[n_tr:n_tr + n_va], 'test': archivos[n_tr + n_va:]}
    for p, lista in partes.items():
        os.makedirs(f'{DIV}/{p}/{c}', exist_ok=True)
        for a in lista:
            shutil.copy(a, f'{DIV}/{p}/{c}/')
for p in ('train', 'val', 'test'):
    print(p, {c: len(os.listdir(f'{DIV}/{p}/{c}')) for c in CLASES})

cargar = lambda p, mezclar: keras.utils.image_dataset_from_directory(
    f'{DIV}/{p}', image_size=(IMG, IMG), batch_size=LOTE, shuffle=mezclar, seed=SEMILLA,
    class_names=CLASES, interpolation='area')
train_ds, val_ds, test_ds = cargar('train', True), cargar('val', False), cargar('test', False)

aumento = keras.Sequential([
    layers.RandomRotation(0.5, fill_mode='nearest'),     # hasta ±180°
    layers.RandomZoom((-0.15, 0.2), fill_mode='nearest'),
    layers.RandomTranslation(0.08, 0.08, fill_mode='nearest'),
    layers.RandomFlip('horizontal'),
    layers.RandomContrast(0.3),
    layers.RandomBrightness(0.25, value_range=(0, 255)),
], name='aumento')

AUTO = tf.data.AUTOTUNE
train_aug = train_ds.map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTO).prefetch(AUTO)
val_ds, test_ds = val_ds.prefetch(AUTO), test_ds.prefetch(AUTO)

# Pesos por clase por si el dataset quedó desbalanceado
n_tr = np.array([len(os.listdir(f'{DIV}/train/{c}')) for c in CLASES])
PESOS = {i: float(n_tr.sum() / (len(CLASES) * n)) for i, n in enumerate(n_tr)}
print("Pesos por clase:", PESOS)

In [ ]:
# Cómo se ve la aumentación (figura para el informe)
x, _ = next(iter(train_ds))
plt.figure(figsize=(12, 2.4))
for i in range(8):
    plt.subplot(1, 8, i + 1); plt.imshow(aumento(x[:1], training=True)[0].numpy().astype('uint8')); plt.axis('off')
plt.suptitle('Aumentación de datos (misma imagen)'); plt.tight_layout()
plt.savefig(f'{FIGURAS}/aumentacion.png', dpi=150); plt.show()

## 4. Modelo: MobileNetV2 preentrenado en ImageNet
La normalización a [-1, 1] va dentro del modelo, así que la Raspberry le pasa la imagen RGB en 0-255 tal cual.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

entrada = keras.Input((IMG, IMG, 3), name='imagen_rgb_0_255')
x = layers.Rescaling(1 / 127.5, offset=-1, name='normalizacion')(entrada)
x = base(x, training=False)                 # BatchNorm congelado también en el fine-tuning
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salida = layers.Dense(len(CLASES), activation='softmax', name='clases')(x)
modelo = keras.Model(entrada, salida, name='torreta_mobilenetv2')
modelo.summary()

## 5. Fase 1: entrenar solo la cabeza

In [ ]:
modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
      keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2)]
h1 = modelo.fit(train_aug, validation_data=val_ds, epochs=EPOCAS_1, class_weight=PESOS, callbacks=cb)

## 6. Fase 2: fine-tuning de las últimas capas de MobileNetV2

In [ ]:
base.trainable = True
for capa in base.layers[:-40]:              # solo se ajustan las ~40 capas finales
    capa.trainable = False
modelo.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = modelo.fit(train_aug, validation_data=val_ds, epochs=EPOCAS_1 + EPOCAS_2, initial_epoch=len(h1.epoch),
                class_weight=PESOS, callbacks=cb)

In [ ]:
# Curvas de entrenamiento (figura para el informe)
hist = {k: h1.history[k] + h2.history[k] for k in ('accuracy', 'val_accuracy', 'loss', 'val_loss')}
fin1 = len(h1.epoch)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, m, t in ((ax[0], 'accuracy', 'Exactitud'), (ax[1], 'loss', 'Pérdida')):
    a.plot(hist[m], label='entrenamiento'); a.plot(hist['val_' + m], label='validación')
    a.axvline(fin1 - 0.5, ls='--', c='gray'); a.text(fin1 - 0.4, a.get_ylim()[1] * 0.97, ' fine-tuning', va='top', fontsize=9)
    a.set_title(t); a.set_xlabel('época'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{FIGURAS}/curvas_entrenamiento.png', dpi=150); plt.show()

## 7. Evaluación en el conjunto de prueba (imágenes que el modelo nunca vio)

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(modelo.predict(test_ds), axis=1)
print(f"Exactitud en prueba (Keras): {np.mean(y_true == y_pred):.4f}\n")
print(classification_report(y_true, y_pred, target_names=CLASES, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax, cmap='Blues', colorbar=False)
plt.xticks(rotation=45); plt.title('Matriz de confusión (prueba)')
plt.tight_layout(); plt.savefig(f'{FIGURAS}/matriz_confusion.png', dpi=150); plt.show()
modelo.save('/content/modelo_torreta.keras')

## 8. Conversión a TensorFlow Lite (cuantización de rango dinámico)

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = conv.convert()
with open('/content/modelo_torreta.tflite', 'wb') as f:
    f.write(tflite)
with open('/content/etiquetas.txt', 'w') as f:
    f.write('\n'.join(CLASES))
print(f"Keras: {os.path.getsize('/content/modelo_torreta.keras') / 1e6:.1f} MB  ->  "
      f"TFLite: {len(tflite) / 1e6:.1f} MB")

## 9. Validar el `.tflite` con el mismo preprocesamiento de la Raspberry
Se lee con `cv2.imread` (BGR), se convierte a RGB y se redimensiona con `INTER_AREA`, igual que `software/clasificador.py`.

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter
interp = Interpreter(model_path='/content/modelo_torreta.tflite')
interp.allocate_tensors()
ent, sal = interp.get_input_details()[0], interp.get_output_details()[0]

aciertos, total, tiempos = 0, 0, []
pred_tfl, real_tfl = [], []
for i, c in enumerate(CLASES):
    for a in glob.glob(f'{DIV}/test/{c}/*'):
        img = cv2.cvtColor(cv2.imread(a), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG, IMG), interpolation=cv2.INTER_AREA)
        interp.set_tensor(ent['index'], img[None].astype(np.float32))
        t0 = time.perf_counter(); interp.invoke(); tiempos.append((time.perf_counter() - t0) * 1000)
        p = int(np.argmax(interp.get_tensor(sal['index'])[0]))
        pred_tfl.append(p); real_tfl.append(i)
acc_tfl = np.mean(np.array(pred_tfl) == np.array(real_tfl))
print(f"Exactitud en prueba (TFLite, flujo de la Raspberry): {acc_tfl:.4f}")
print(f"Latencia en Colab (CPU): {np.mean(tiempos):.1f} ms por imagen (en la Pi 4 se mide con torreta_ia.py --solo-vision)")

## 10. Descargar el modelo y las figuras

In [ ]:
with zipfile.ZipFile('/content/modelo_torreta.zip', 'w') as z:
    z.write('/content/modelo_torreta.tflite', 'modelo/modelo_torreta.tflite')
    z.write('/content/etiquetas.txt', 'modelo/etiquetas.txt')
    for f in glob.glob(f'{FIGURAS}/*.png'):
        z.write(f, 'figuras/' + os.path.basename(f))
from google.colab import files
files.download('/content/modelo_torreta.zip')
# En la Raspberry: copia modelo/ dentro de software/  (software/modelo/modelo_torreta.tflite)